CHECKPOINT 02: Assistente de IA Generativa com Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Emanuel Italo (RM 561337)
- Enzo Monteiro Maciel (RM 563734)
- Gabriel Bebé Silva (RM 562012)
- Matheus de Almeida Sousa (RM 563557)
- Paulo Estalise (RM 563811)

**Tema escolhido:** 2. Suporte técnico de IoT — diagnóstico de problemas comuns com ESP32, Arduino, Wi-Fi e sensores

---

Nesta avaliação o grupo vai aprofundar o assistente construído na atividade anterior, em 4 etapas (e 1 bônus):

1. Assistente com guardrails (restrições no `system`)
2. Comparação com temperatura
3. Chat com memória
4. Interface web com Gradio e controle de temperatura
5. API com sessões independentes

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.  
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [9]:
%pip install google-genai gradio python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
from google import genai
from google.genai import types


try:
    from google.colab import userdata
    EM_COLAB = True
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.getcwd(), ".env"))
    EM_COLAB = False

In [11]:
if EM_COLAB:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
else:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY no .env (ou nos Secrets do Colab)")

GEMINI_API_KEY ok


In [12]:
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [13]:

SYSTEM_PROMPT = """Você é o IoT Help, um assistente de suporte técnico de IoT.
Ajude o usuário a diagnosticar problemas comuns com ESP32, Arduino, conexão Wi-Fi e sensores
(ex.: DHT11/DHT22, HC-SR04, PIR, LDR), incluindo ligação elétrica, alimentação, bibliotecas e código.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Quando fizer sentido, sugira um teste simples para o usuário confirmar a causa do problema.
Se o problema envolver a rede elétrica da casa (127V/220V), recomende procurar um eletricista.
Se a pergunta não tiver relação com suporte técnico de IoT, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com guardrails

Na atividade anterior o grupo testou personalidades diferentes.
Aqui o foco é o **guardrail**: usar o `system` para restringir o assistente a responder **apenas** perguntas do tema.

A mesma pergunta será enviada com três versões do `system`:
- sem restrição
- com restrição de tema
- com restrição de tema e de formato

Também será testada uma pergunta **fora do escopo** para ver se o assistente respeita o guardrail.

In [14]:
def perguntar_gemini(pergunta, system, temperatura=1.0):
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [15]:

versoes_system = {
    "Sem restrição": "Você é um assistente prestativo. Responda em português.",

    "Com restrição de tema": """Você é um assistente de suporte técnico de IoT (ESP32, Arduino, Wi-Fi e sensores).
Se a pergunta não for sobre suporte técnico de IoT, diga que não pode ajudar.
Responda em português.""",

    "Com restrição de tema e formato": """Você é um assistente de suporte técnico de IoT (ESP32, Arduino, Wi-Fi e sensores).
Se a pergunta não for sobre suporte técnico de IoT, diga que não pode ajudar.
Responda em português, em no máximo 3 frases, sem usar listas ou tópicos.""",
}

pergunta_dentro = "Meu ESP32 não conecta no Wi-Fi e fica reiniciando sozinho. O que pode ser?"
pergunta_fora   = "Me passa uma receita de bolo de chocolate?"

for nome, system in versoes_system.items():
    print(f"\n{'='*55}")
    print(f"  {nome}")
    print(f"{'='*55}")
    print(f"\n[Dentro do escopo] {pergunta_dentro}")
    print(perguntar_gemini(pergunta_dentro, system))
    print(f"\n[Fora do escopo]   {pergunta_fora}")
    print(perguntar_gemini(pergunta_fora, system))


  Sem restrição

[Dentro do escopo] Meu ESP32 não conecta no Wi-Fi e fica reiniciando sozinho. O que pode ser?
Esse é um problema clássico no ESP32. Quando ele tenta conectar no Wi-Fi e reinicia sozinho, geralmente ocorre um **Loop de Reinicialização (Restart Loop)** causado por uma queda de energia ou um erro de software (exceção).

Aqui estão as causas mais comuns e como resolver, da mais provável para a menos provável:

### 1. Problema de Alimentação (A causa No 1)
O ESP32 consome muita energia (picos de até 500mA) quando o rádio Wi-Fi é ligado. 
* **Cabo USB ruim ou porta USB fraca:** Portas USB de computadores mais antigos (USB 2.0) ou hubs USB sem alimentação própria podem não fornecer corrente suficiente. Quando o Wi-Fi liga, a tensão cai e o ESP32 reinicia.
* **Solução:** Tente usar um cabo USB de boa qualidade, conecte em uma porta USB 3.0 (azul) ou, idealmente, alimente a placa com uma fonte externa de 5V 2A (como carregador de celular) conectada ao pino 5V/VIN e GND.

### 2

**Observações do grupo (Etapa 1):**

- O assistente sem restrição respondeu a pergunta fora do escopo?
- Com o guardrail de tema, ele recusou? A recusa foi educada?
- A restrição de formato mudou o estilo da resposta?

(responda aqui)

---
## Etapa 2: Comparação com temperatura

Aqui o grupo compara o efeito da **temperatura** nas respostas do Gemini.

- `temperature=0.0` — o modelo escolhe sempre o token mais provável
- `temperature=1.0` — o modelo amostra com mais aleatoriedade

Cada pergunta é enviada **duas vezes** com cada valor de temperatura, para ver se as respostas se repetem ou variam.

In [16]:
def comparar_temperaturas(pergunta, temperaturas=(0.0, 1.0), repeticoes=2):
    print(f"\nPERGUNTA: {pergunta}")
    for temp in temperaturas:
        print(f"\n--- temperature={temp} ---")
        for n in range(1, repeticoes + 1):
            print(f"[Gemini, tentativa {n}] {perguntar_gemini(pergunta, SYSTEM_PROMPT, temperatura=temp)}")
    print("\n" + "=" * 55)

In [17]:

perguntas = [
    "Por que meu sensor DHT11 retorna NaN na leitura com o ESP32?",
    "Como alimentar um ESP32 com bateria sem queimar a placa?",
]

for p in perguntas:
    comparar_temperaturas(p)


PERGUNTA: Por que meu sensor DHT11 retorna NaN na leitura com o ESP32?

--- temperature=0.0 ---
[Gemini, tentativa 1] O erro "NaN" geralmente indica um problema de conexão física, como fios invertidos ou mal conectados nos pinos de dados, VCC e GND. Verifique também se você está usando a biblioteca correta (`DHT sensor library` da Adafruit) e se definiu o pino e o modelo do sensor corretamente no código. Outra causa comum é a falta de um resistor de pull-up (geralmente de 4k7 a 10k ohms) entre o pino de dados e o 3.3V. Por fim, lembre-se de que o ESP32 opera em 3.3V, embora alguns módulos do DHT11 funcionem melhor alimentados em 5V. Tente trocar o pino GPIO usado no código para descartar uma falha na porta do microcontrolador.
[Gemini, tentativa 2] O erro "NaN" no sensor DHT11 geralmente indica um problema de conexão física, pinagem incorreta ou falta do resistor de pull-up de 4.7kΩ a 10kΩ entre o pino de dados e o VCC. Verifique também se você alimentou o sensor com 5V (ou 3.3V, depe

**Observações do grupo (Etapa 2):**

- Com `temperature=0.0`, as duas tentativas deram respostas iguais ou muito parecidas?
- Com `temperature=1.0`, as respostas ficaram mais variadas?
- Qual temperatura vocês consideram melhor para o tema?

(responda aqui)

---
## Etapa 3: Chat com memória

O chat guarda o histórico da conversa e envia tudo para o Gemini a cada mensagem.
É isso que faz o assistente "lembrar" do que foi dito antes.

Comandos:
- `sair` — encerra o chat
- `limpar` — apaga o histórico
- `historico` — mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [18]:
def responder_chat(historico):
    
    conteudos = []
    for msg in historico:
        if msg["role"] == "system":
            continue  
        papel = "model" if msg["role"] == "assistant" else "user"
        conteudos.append({"role": papel, "parts": [{"text": msg["content"]}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [4]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat iniciado com o Gemini")
print("Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens: {len(historico)})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    texto = responder_chat(historico)
    historico.append({"role": "assistant", "content": texto})

    print(f"\n[Gemini] {texto}\n")

NameError: name 'SYSTEM_PROMPT' is not defined

In [1]:
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

NameError: name 'historico' is not defined

**Observações do grupo (Etapa 3):**

- O assistente "lembrou" do que foi dito antes na conversa?
- Depois do comando `limpar`, ele ainda lembrava? Por quê?
- Por que o `system` vai em `system_instruction` e não junto com as mensagens?

(responda aqui)

---
## Etapa 4: Interface web com Gradio e controle de temperatura

A interface tem um **slider de temperatura** para controlar a criatividade das respostas em tempo real.

In [ ]:
import gradio as gr

def texto_da_mensagem(conteudo):
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, temperatura):
    conteudos = []
    for msg in historico_gradio:
        papel = "model" if msg["role"] == "assistant" else "user"
        conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
    conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [ ]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[
        gr.Slider(minimum=0.0, maximum=1.0, step=0.1, value=0.7, label="Temperatura"),
    ],
    title="IoT Help: Suporte Técnico de IoT",
    description="Descreva o problema com seu ESP32, Arduino, Wi-Fi ou sensor e receba um diagnóstico passo a passo.",
    examples=[
        ["Meu Arduino não aparece na porta COM do computador. O que faço?", 0.2],
        ["O sensor HC-SR04 só retorna distância 0. Qual pode ser a causa?",  0.2],
        ["O ESP32 desconecta do Wi-Fi depois de algumas horas ligado.",      0.7],
    ],
).launch(share=True)

**Observações do grupo (Etapa 4):**

- Tire um print da interface com o slider visível e coloque no README do repositório.
- Teste a mesma pergunta com temperatura 0.0 e 1.0 na interface. O que mudou?
- Qual temperatura funcionou melhor para o tema?

(responda aqui)

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5: API com sessões independentes

Na atividade anterior a API não guardava histórico entre chamadas.
Aqui cada `session_id` tem seu próprio histórico — duas sessões paralelas não compartilham contexto.

In [ ]:
%%writefile app_avaliacao.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from google import genai
from google.genai import types

# Chave e system prompt vêm de variáveis de ambiente (nunca escreva a chave no código)
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODELO = os.environ.get("MODELO_GEMINI", "gemini-3.5-flash-lite")
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

# Histórico por sessão (já no formato do Gemini)
sessoes: dict[str, list] = {}

class Pergunta(BaseModel):
    session_id: str
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    historico = sessoes.setdefault(pergunta.session_id, [])
    historico.append({"role": "user", "parts": [{"text": pergunta.mensagem}]})

    resposta = client.models.generate_content(
        model=MODELO,
        contents=historico,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
        ),
    )
    texto = resposta.text

    historico.append({"role": "model", "parts": [{"text": texto}]})

    return {"session_id": pergunta.session_id, "resposta": texto}

@app.delete("/sessao/{session_id}")
def limpar_sessao(session_id: str):
    sessoes.pop(session_id, None)
    return {"status": "sessão apagada", "session_id": session_id}

In [ ]:
%pip install fastapi uvicorn -q

import os, subprocess, sys, time
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["MODELO_GEMINI"]  = MODELO_GEMINI
os.environ["SYSTEM_PROMPT"]  = SYSTEM_PROMPT

# sys.executable -m uvicorn usa o mesmo Python do notebook (funciona no Windows e no Colab)
servidor = subprocess.Popen([sys.executable, "-m", "uvicorn", "app_avaliacao:app", "--port", "8000"])
time.sleep(5)
print("Servidor rodando em http://localhost:8000")

In [ ]:
import requests

# Sessão A — apresenta o nome
requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Meu nome é Ana."})
r_a = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Qual é o meu nome?"})

# Sessão B — não apresentou o nome
r_b = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_b", "mensagem": "Qual é o meu nome?"})

print("Sessão A:", r_a.json()["resposta"])
print("Sessão B:", r_b.json()["resposta"])

In [ ]:
servidor.terminate()
print("Servidor encerrado.")

**Observações do grupo (Etapa 5 — Bônus):**

- A sessão A respondeu o nome corretamente? E a sessão B?
- O que mudou nessa API em relação à da atividade anterior?
- Cite um caso de uso real onde sessões independentes seriam necessárias.

(responda aqui)